# Full Chain Training + Inference (Unified Pipeline)

This notebook runs the complete 12-stage chain in one pipeline:
group-classifier -> group-splitter -> endpoint-regression -> event-splitter -> pion-stop -> positron-angle,
with training and inference for each model in sequence.

## 1) Environment Setup

In [1]:
from datetime import datetime
from pathlib import Path
from typing import Any

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

from pioneerml.integration.zenml import load_step_output
from pioneerml.integration.zenml import utils as zenml_utils

PROJECT_ROOT = Path(zenml_utils.find_project_root())
zenml_utils.setup_zenml_for_notebook(root_path=PROJECT_ROOT, use_in_memory=True)

MODEL_CHAIN = (
    "group_classifier",
    "group_splitter",
    "endpoint_regression",
    "event_splitter",
    "pion_stop",
    "positron_angle",
)


Using ZenML repository root: /workspace
Ensure this is the top-level of your repo (.zen must live here).


## 2) Select Input Data and Output Workspace

By default this notebook builds an aligned 8-row shard for quick iteration.
Set `USE_SMALL_SUBSET = False` to run against the full source parquet.

In [2]:
def _select_aligned_row_indices(
    *,
    main_file: Path,
    count: int = 8,
    min_hits: int = 12,
    min_positron_hits: int = 2,
) -> list[int]:
    table = pq.read_table(main_file, columns=["hits_pdg_id", "hits_particle_mask"])
    pdg_lists = table["hits_pdg_id"].to_pylist() if "hits_pdg_id" in table.column_names else [None] * table.num_rows
    mask_lists = table["hits_particle_mask"].to_pylist()

    lengths = np.asarray([len(v) if v is not None else 0 for v in mask_lists], dtype=np.int64)
    positron_counts = np.asarray(
        [int(sum(int(x) == -11 for x in (row or []))) for row in pdg_lists],
        dtype=np.int64,
    )

    eligible = np.flatnonzero((lengths >= int(min_hits)) & (positron_counts >= int(min_positron_hits)))
    if eligible.size == 0:
        raise RuntimeError(
            f"No rows with >= {min_hits} hits and >= {min_positron_hits} positron hits found in {main_file}."
        )
    order = np.argsort(lengths[eligible], kind="stable")
    chosen = eligible[order][: int(count)]
    if chosen.size == 0:
        raise RuntimeError(f"Failed to select aligned rows from {main_file}.")
    return [int(v) for v in chosen.tolist()]


def _write_aligned_subset(*, src: Path, dst: Path, row_indices: list[int]) -> None:
    table = pq.read_table(src)
    index_arr = pa.array([int(v) for v in row_indices], type=pa.int64())
    pq.write_table(table.take(index_arr), dst)


USE_SMALL_SUBSET = False
SMALL_ROW_COUNT = 8

src_main = PROJECT_ROOT / "data" / "ml_output_000.parquet"
workspace_dir = PROJECT_ROOT / "artifacts" / "full_chain_notebook"
models_dir = workspace_dir / "models"
preds_dir = workspace_dir / "preds"
workspace_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)
preds_dir.mkdir(parents=True, exist_ok=True)

if USE_SMALL_SUBSET:
    main_path = workspace_dir / "ml_output_000_small.parquet"
    selected_rows = _select_aligned_row_indices(main_file=src_main, count=int(SMALL_ROW_COUNT))
    _write_aligned_subset(src=src_main, dst=main_path, row_indices=selected_rows)
else:
    main_path = src_main

main_sources = [str(main_path)]

pred_paths = {
    "group_classifier": preds_dir / "group_classifier_preds.parquet",
    "group_splitter": preds_dir / "group_splitter_preds.parquet",
    "endpoint_regression": preds_dir / "endpoint_regressor_preds.parquet",
    "event_splitter": preds_dir / "event_splitter_preds.parquet",
    "pion_stop": preds_dir / "pion_stop_preds.parquet",
    "positron_angle": preds_dir / "positron_angle_preds.parquet",
}

for p in pred_paths.values():
    if p.exists():
        p.unlink()

print("main_path:", main_path)
print("workspace_dir:", workspace_dir)


main_path: /workspace/data/ml_output_000.parquet
workspace_dir: /workspace/artifacts/full_chain_notebook


## 3) Reusable Config Helpers

In [3]:
from pioneerml.plugin.runtime import ensure_plugins_loaded
ensure_plugins_loaded()

from copy import deepcopy

from pioneerml_base_plugin.full_training_chain.pipeline import full_chain_pipeline, load_config
from pioneerml_base_plugin.utils.config_loader import with_export_output, with_loader_sources, with_writer_output


## 4) Build Full Chain Config

In [4]:
HPO_TRIALS = 10
MAX_EPOCHS = 10
TRAIN_BATCH_SIZE = None
INFERENCE_BATCH_SIZE = 32
SAMPLE_FRACTION = 0.05
EVALUATE_ENABLED = True

input_specs = {
    "group_classifier": {
        "main_sources": list(main_sources),
        "optional_sources_by_name": {},
    },
    "group_splitter": {
        "main_sources": list(main_sources),
        "optional_sources_by_name": {"group_probs": [str(pred_paths["group_classifier"])]},
    },
    "endpoint_regression": {
        "main_sources": list(main_sources),
        "optional_sources_by_name": {
            "group_probs": [str(pred_paths["group_classifier"])],
            "group_splitter": [str(pred_paths["group_splitter"])],
        },
    },
    "event_splitter": {
        "main_sources": list(main_sources),
        "optional_sources_by_name": {
            "group_probs": [str(pred_paths["group_classifier"])],
            "group_splitter": [str(pred_paths["group_splitter"])],
            "endpoint": [str(pred_paths["endpoint_regression"])],
        },
    },
    "pion_stop": {
        "main_sources": list(main_sources),
        "optional_sources_by_name": {
            "group_probs": [str(pred_paths["group_classifier"])],
            "group_splitter": [str(pred_paths["group_splitter"])],
            "endpoint": [str(pred_paths["endpoint_regression"])],
            "event_splitter": [str(pred_paths["event_splitter"])],
        },
    },
    "positron_angle": {
        "main_sources": list(main_sources),
        "optional_sources_by_name": {
            "group_probs": [str(pred_paths["group_classifier"])],
            "group_splitter": [str(pred_paths["group_splitter"])],
            "endpoint": [str(pred_paths["endpoint_regression"])],
            "event_splitter": [str(pred_paths["event_splitter"])],
            "pion_stop": [str(pred_paths["pion_stop"])],
        },
    },
}

export_meta = {
    "group_classifier": ("groupclassifier", "groupclassifier"),
    "group_splitter": ("groupsplitter", "groupsplitter"),
    "endpoint_regression": ("endpoint_regressor", "endpoint_regressor"),
    "event_splitter": ("event_splitter", "event_splitter"),
    "pion_stop": ("pion_stop_regression", "pion_stop"),
    "positron_angle": ("positron_angle_regression", "positron_angle"),
}

base_config = load_config()
pipeline_config = {"common": dict(base_config.get("common") or {})}

for model_key in MODEL_CHAIN:
    spec = dict(input_specs[model_key])
    export_subdir, export_prefix = export_meta[model_key]

    train_cfg = deepcopy(base_config[model_key]["training"])
    train_cfg = with_loader_sources(
        train_cfg,
        main_sources=spec["main_sources"],
        optional_sources_by_name=spec["optional_sources_by_name"],
    )
    train_cfg = with_export_output(
        train_cfg,
        export_dir=str(models_dir / export_subdir),
        filename_prefix=export_prefix,
    )

    train_cfg["hpo"]["hpo"]["config"]["n_trials"] = int(HPO_TRIALS)
    train_cfg["hpo"]["loader_manager"]["config"]["defaults"]["config"]["sample_fraction"] = float(SAMPLE_FRACTION)
    train_cfg["train"]["loader_manager"]["config"]["defaults"]["config"]["sample_fraction"] = float(SAMPLE_FRACTION)
    train_cfg["hpo"]["trainer"]["config"]["trainer_kwargs"]["max_epochs"] = int(min(10, max(1, MAX_EPOCHS)))
    train_cfg["train"]["trainer"]["config"]["trainer_kwargs"]["max_epochs"] = int(MAX_EPOCHS)
    train_cfg["evaluate"]["enabled"] = bool(EVALUATE_ENABLED)

    infer_cfg = deepcopy(base_config[model_key]["inference"])
    infer_cfg = with_loader_sources(
        infer_cfg,
        main_sources=spec["main_sources"],
        optional_sources_by_name=spec["optional_sources_by_name"],
    )
    infer_cfg = with_writer_output(
        infer_cfg,
        output_dir=str(pred_paths[model_key].parent),
        output_path=str(pred_paths[model_key]),
    )
    infer_cfg["inference"]["loader_manager"]["config"]["defaults"]["config"]["batch_size"] = int(INFERENCE_BATCH_SIZE)

    pipeline_config[model_key] = {"training": train_cfg, "inference": infer_cfg}


## 5) Run Full Chain Pipeline

In [5]:
run = full_chain_pipeline.with_options(enable_cache=False)(
    pipeline_config=pipeline_config,
)


Initiating a new run for the pipeline: full_chain_pipeline.
Caching is disabled by default for full_chain_pipeline.
Using user: default
Using stack: default
  deployer: default
  artifact_store: default
  orchestrator: default
You can visualize your pipeline runs in the ZenML Dashboard. In order to try it locally, please run zenml login --local.
Step run_training_stage_step has started.


[I 2026-03-26 01:52:06,928] A new study created in RDB with name: group_classifier_hpo


[run_training_stage_step] GPU available: True (cuda), used: True
[run_training_stage_step] TPU available: False, using: 0 TPU cores
[run_training_stage_step] You are using a CUDA device ('NVIDIA GeForce RTX 5070') that has Tensor Cores. To properly utilize them, you should set torch.set_float32_matmul_precision('medium' | 'high') which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
[run_training_stage_step] LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type                  ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ GroupClassifierStereo │  1.4 M │ train │     0 │
│ 1 │ loss_fn │ BCEWithLogitsLoss     │      0 │ train │     0 │
└───┴─────────┴───────────────────────┴────────┴───────┴───────┘

Trainable params: 1.4 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.4 M                                                                                                
Total estimated model params size (MB): 5                                                                          
Modules in train mode: 58                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

[run_training_stage_step_6] GPU available: True (cuda), used: True
[run_training_stage_step_6] TPU available: False, using: 0 TPU cores
[run_training_stage_step_6] LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type                ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ PositronAngleModel  │  1.4 M │ train │     0 │
│ 1 │ loss_fn │ QuantileAngularLoss │      0 │ train │     0 │
└───┴─────────┴─────────────────────┴────────┴───────┴───────┘

Trainable params: 1.4 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.4 M                                                                                                
Total estimated model params size (MB): 5                                                                          
Modules in train mode: 46                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

[run_training_stage_step_6] No materializer is registered for type <class 'pioneerml.integration.pytorch.modules.graph_lightning_module.GraphLightningModule'>, so the default Pickle materializer was used. Pickle is not production ready and should only be used for prototyping as the artifacts cannot be loaded when running with a different Python version. Please consider implementing a custom materializer for type <class 'pioneerml.integration.pytorch.modules.graph_lightning_module.GraphLightningModule'> according to the instructions at https://docs.zenml.io/concepts/artifacts/materializers
Step run_training_stage_step_6 has finished in 1m8s.
Step run_cleanup_step_11 has started.
Step run_cleanup_step_11 has finished in 1.109s.
Step run_inference_stage_step_6 has started.
[run_inference_stage_step_6] No materializer is registered for type <class 'pioneerml.integration.pytorch.model_handles.torchscript_model_handle.TorchScriptModelHandle'>, so the default Pickle materializer was used. Pic

## 6) Inspect Stage Outputs

In [6]:
def _step_name(base: str, idx1: int) -> str:
    return base if idx1 == 1 else f"{base}_{idx1}"

print("run_id:", run.id)

for idx, model_key in enumerate(MODEL_CHAIN, start=1):
    train_step = _step_name("run_training_stage_step", idx)
    infer_step = _step_name("run_inference_stage_step", idx)

    train_out = load_step_output(run, train_step)
    infer_out = load_step_output(run, infer_step)

    export_payload = (dict(train_out or {}).get("export") if isinstance(train_out, dict) else None) or {}
    inference_payload = (dict(infer_out or {}).get("inference") if isinstance(infer_out, dict) else None) or {}

    pred_paths_out = list(dict(inference_payload or {}).get("predictions_paths") or [])
    if not pred_paths_out and isinstance(inference_payload, dict) and inference_payload.get("predictions_path"):
        pred_paths_out = [str(inference_payload["predictions_path"])]

    print(f"[{model_key}] export:", export_payload)
    print(f"[{model_key}] predictions:", pred_paths_out)

final_pred_path = pred_paths["positron_angle"]
if final_pred_path.exists():
    tbl = pq.read_table(final_pred_path)
    print("final_positron_rows:", tbl.num_rows)
    print("final_positron_columns:", tbl.column_names)
else:
    print("Final positron-angle prediction file not found yet:", final_pred_path)


run_id: b7f14e06-dd44-4417-8fc4-48ffbfc1f5cf
[group_classifier] export: {'torchscript_path': '/workspace/artifacts/full_chain_notebook/models/groupclassifier/groupclassifier_20260326_015241_torchscript.pt', 'metadata_path': '/workspace/artifacts/full_chain_notebook/models/groupclassifier/groupclassifier_20260326_015241_meta.json', 'export_type': 'script', 'exporter_type': 'torchscript'}
[group_classifier] predictions: ['/workspace/artifacts/full_chain_notebook/preds/group_classifier_preds.parquet']
[group_splitter] export: {'torchscript_path': '/workspace/artifacts/full_chain_notebook/models/groupsplitter/groupsplitter_20260326_015316_torchscript.pt', 'metadata_path': '/workspace/artifacts/full_chain_notebook/models/groupsplitter/groupsplitter_20260326_015316_meta.json', 'export_type': 'script', 'exporter_type': 'torchscript'}
[group_splitter] predictions: ['/workspace/artifacts/full_chain_notebook/preds/group_splitter_preds.parquet']
[endpoint_regression] export: {'torchscript_path': 